In [11]:
import geopandas as gpd
import geopandas as gpd
from shapely.ops import unary_union, polygonize
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import linregress
import numpy as np

In [12]:
west = [
    "NP-P37", 
    "Upstream_Taylor",
    "ENPLM",
    "ENPTR",
    "NP-146",
    "NP-CHP",
    "JBTS",
    "ENPTC",
    "NP-OL", 
    "SWEVER5A",
    "MUD_CRKM",
]

central = [
    "EVER7", 
    "CT50R", 
    "EVER6", 
    "CT27R", 
    "EVER8",
    "ENPLS",
    "ENPDK",
    "ENPBS",
    "ENPHC",
    "NP-EV6",
    "NP-EP1"
]

east = [
     "SWEVER1",
     "S197_H",
     "S197_T",
     "S20_H",
     "S20_T"
]

In [13]:
### read EDEN data ##
hydro_df = pd.read_csv(
    '/Volumes/Malonelab/Research/Whitezone/Hydro_Salinity/Hydro/Merged/annual_summary_merged.csv'
)

# Convert types
hydro_df['year'] = hydro_df['year'].astype(int)
hydro_df = hydro_df[hydro_df['year'] != 2025]
hydro_df['mean_value_NAVD88'] = pd.to_numeric(hydro_df['mean_value'], errors='coerce') * 30.48

In [14]:
# =========================
# Read datum file
# =========================
datum = pd.read_csv(
    "/Volumes/Malonelab/Research/Whitezone/Hydro_Salinity/Hydro/Merged/WZ_hydro_sites_latlon_deg.csv"
)

# =========================
# Clean matching columns
# =========================
hydro_df['source_file'] = hydro_df['source_file'].astype(str).str.strip()
datum['Name'] = datum['Name'].astype(str).str.strip()

# =========================
# Merge on site name
# =========================
df = hydro_df.merge(
    datum[['Name', 'Datum', 'Conversion']],
    left_on='source_file',
    right_on='Name',
    how='left'
)

# =========================
# Ensure numeric types
# =========================
df['mean_value_NAVD88'] = pd.to_numeric(df['mean_value_NAVD88'], errors='coerce')
df['Conversion'] = pd.to_numeric(df['Conversion'], errors='coerce')
df['Datum'] = pd.to_numeric(df['Datum'], errors='coerce')

# =========================
# Apply conversion rule
# If Datum == 29 → add (Conversion * 30.48)
# Otherwise keep original NAVD88 value
# =========================
df['mean_value_NAVD88_corrected'] = np.where(
    df['Datum'] == 29,
    df['mean_value_NAVD88'] + df['Conversion'] * 30.48,
    df['mean_value_NAVD88']
)

# =========================
# Optional: drop helper column
# =========================
df = df.drop(columns=['Name'])

print("Conversion applied. New column: mean_value_NAVD88_corrected")
combined_df = df

Conversion applied. New column: mean_value_NAVD88_corrected


In [15]:
# =========================
# Time span per East, West, Central site
# =========================

# Make sure combined_df is your main DataFrame with 'source_file', 'year', 'mean_value_NAVD88'

def compute_site_span(df, site_list, label):
    temp_df = df[df["source_file"].isin(site_list)].copy()
    temp_df["year"] = pd.to_numeric(temp_df["year"], errors="coerce")
    temp_df = temp_df.dropna(subset=["year", "mean_value_NAVD88"])
    
    site_span = (
        temp_df.groupby("source_file")["year"]
        .agg(
            start_year="min",
            end_year="max",
            n_years="nunique"
        )
        .reset_index()
    )
    print(f"=== {label} Site Time Spans ===")
    print(site_span)
    return site_span

# Compute for East
east_span = compute_site_span(combined_df, east, "East")

# Compute for West
west_span = compute_site_span(combined_df, west, "West")

# Compute for Central
central_span = compute_site_span(combined_df, central, "Central")


=== East Site Time Spans ===
  source_file  start_year  end_year  n_years
0      S197_H        1991      2024       33
1      S197_T        1991      2024       29
2       S20_H        1991      2024       34
3       S20_T        1991      2024       34
4     SWEVER1        1991      2024       34
=== West Site Time Spans ===
        source_file  start_year  end_year  n_years
0             ENPLM        1993      2024       32
1             ENPTC        1993      2024       32
2             ENPTR        1993      2024       32
3              JBTS        1992      2024       33
4          MUD_CRKM        1995      2024       30
5            NP-146        1994      2024       31
6            NP-CHP        1978      2024       47
7             NP-OL        1996      2024       29
8            NP-P37        1953      2024       72
9          SWEVER5A        1985      2024       40
10  Upstream_Taylor        1999      2024       26
=== Central Site Time Spans ===
   source_file  start_year  

In [16]:
## Analyze the trend between 2013 to 2024
# Filter years 2013–2024
test_df = combined_df[(combined_df["year"] >= 2014) & (combined_df["year"] <= 2024)].copy()

# Assign region
def assign_region(station):
    if station in west:
        return "west"
    elif station in central:
        return "central"
    elif station in east:
        return "east"
    else:
        return None  # ignore stations not listed

test_df["region"] = test_df["source_file"].apply(assign_region)

# Subtract station mean instead of 2013 baseline
demeaned_list = []
for station, subdf in test_df.groupby("source_file"):
    subdf = subdf.copy()
    mean_val = subdf["mean_value_NAVD88"].mean()
    subdf["demeaned"] = subdf["mean_value_NAVD88"] - mean_val
    demeaned_list.append(subdf)

demeaned_df = pd.concat(demeaned_list)

# Calculate slope per region using demeaned values
trends = {}
for region, subdf in demeaned_df.groupby("region"):
    # Aggregate by year: mean across stations in the region
    yearly_mean = subdf.groupby("year")["demeaned"].mean().reset_index()
    
    slope, intercept, r_value, p_value, std_err = linregress(yearly_mean["year"], yearly_mean["demeaned"])
    
    trends[region] = {
        "slope": slope,
        "p_value": p_value,
        "r2": r_value**2
    }

trend_df = pd.DataFrame(trends).T
print("Demeaned regression trends (2013–2024):")
print(trend_df)


Demeaned regression trends (2013–2024):
            slope   p_value        r2
central  1.100651  0.000670  0.740849
east     1.064582  0.001689  0.683889
west     0.951339  0.000575  0.749249


In [17]:
# Count original rows
original_rows = combined_df.shape[0]

# Remove duplicates: keep first occurrence per station per year
combined_df = combined_df.drop_duplicates(subset=["source_file", "year"])

# Count rows after deduplication
new_rows = combined_df.shape[0]

# Report how many rows were removed
rows_removed = original_rows - new_rows
print(f"Removed {rows_removed} duplicate rows.")

Removed 29 duplicate rows.


In [18]:
%matplotlib agg
### Plot with mean values ####
plt.rcParams["font.family"] = "Times New Roman"
fs = 22

# Filter years 1996–2024
df = combined_df[(combined_df["year"] >= 1991) & (combined_df["year"] <= 2024)]

# Filter only the stations in each region
west_df = df[df["source_file"].isin(west)]
central_df = df[df["source_file"].isin(central)]
east_df = df[df["source_file"].isin(east)]

plt.figure(figsize=(12, 6))

# Define regions and colors
regions = {
    "West": (west_df, "#fee08b"),
    "Central": (central_df, "orange"),
    "East": (east_df, "red")
}

# Plot each region
for name, (subdf, color) in regions.items():
    if subdf.empty:
        continue  # skip if no data

    # Compute yearly stats
    grouped = subdf.groupby("year")["mean_value_NAVD88"]
    mean_vals = grouped.mean()
    std_vals = grouped.std()
    counts = grouped.count()
    se_vals = std_vals / np.sqrt(counts)   # <-- standard error
    # Shaded region: mean ± 2*std
    plt.fill_between(mean_vals.index, mean_vals - 1*se_vals, mean_vals + 1*se_vals,
                     facecolor=color, alpha=0.2, edgecolor='none')

    # Mean line
    plt.plot(mean_vals.index, mean_vals.values, 
             color=color, linewidth=2, label=name)


# Labels and title
plt.xlabel("Year", fontsize=fs)
plt.ylabel("Water Level (cm)", fontsize=fs)
plt.title("Annual Mean Water Level", fontsize=fs)
plt.legend(fontsize=fs, bbox_to_anchor=(1.05, 1), loc="upper left")

# Start year, end year, interval
start_year = 1991
end_year = 2024
interval = 11

# Generate ticks
years = list(np.arange(start_year, end_year+1, interval))  # every 6 years
if years[-1] != end_year:
    years.append(end_year)  # ensure 2024 is included

plt.xticks(ticks=years, labels=years, fontsize=fs)
plt.yticks(fontsize=fs)

# Lock axis range explicitly
plt.xlim(1991, 2024)
plt.ylim(-30, 40)

plt.grid(alpha=0.3)
plt.tight_layout()

# Save figure first, then show
plt.savefig("/Users/bz294/Documents/WhiteZone/Figure/Backup/wl_plot.tif", dpi=300, bbox_inches="tight")
plt.show()


/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_38940/2969302269.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
import pandas as pd

# Dictionary to store stats
region_stats = {}

for name, (subdf, color) in regions.items():
    if subdf.empty:
        continue  # skip if no data

    # Compute yearly mean across stations
    yearly_mean = subdf.groupby("year")["mean_value_NAVD88"].mean()

    n = yearly_mean.count()

    region_stats[name] = {
        "mean": yearly_mean.mean(),
        "std": yearly_mean.std(),
        "se": yearly_mean.std() / (n ** 0.5)
    }

# Convert to DataFrame
region_stats_df = pd.DataFrame(region_stats).T
region_stats_df.index.name = "Region"

print("Summary statistics (mean ± std) for each subregion (1991–2024):")
print(region_stats_df)


Summary statistics (mean ± std) for each subregion (1991–2024):
              mean       std        se
Region                                
West     13.510050  6.740498  1.155986
Central  16.581303  5.898236  1.011539
East     10.374967  5.386519  0.923780


In [9]:
# Filter only sites in your three groups
all_sites = west + central + east
df_sites = combined_df[combined_df["source_file"].isin(all_sites)]

# Time span per site
time_span_per_site = df_sites.groupby("source_file")["year"].agg(["min", "max"]).reset_index()
print(time_span_per_site)

        source_file   min   max
0             CT27R  1990  2024
1             CT50R  1990  2024
2             ENPBS  2000  2024
3             ENPDK  1999  2024
4             ENPHC  1998  2024
5             ENPLM  1993  2024
6             ENPLS  1993  2024
7             ENPTC  1993  2024
8             ENPTR  1993  2024
9             EVER6  1991  2024
10            EVER7  1991  2024
11            EVER8  1991  2024
12             JBTS  1992  2024
13         MUD_CRKM  1995  2024
14           NP-146  1994  2024
15           NP-CHP  1978  2024
16           NP-EP1  1989  2024
17           NP-EV6  1991  2024
18            NP-OL  1996  2024
19           NP-P37  1953  2024
20           S197_H  1991  2024
21           S197_T  1991  2024
22            S20_H  1991  2024
23            S20_T  1991  2024
24          SWEVER1  1991  2024
25         SWEVER5A  1985  2024
26  Upstream_Taylor  1999  2024


In [10]:
# =============================
# Calculate mean and std of annual mean water level for each subregion
# =============================

# Store yearly mean series for each region
yearly_averages = {}

for name, (subdf, color) in regions.items():
    if subdf.empty:
        continue
    grouped = subdf.groupby("year")["mean_value_NAVD88"]
    mean_vals = grouped.mean()
    yearly_averages[name] = mean_vals

# Calculate mean and std across the time series for each region
overall_stats = {}

for name, mean_vals in yearly_averages.items():
    if not mean_vals.empty:
        overall_stats[name] = {
            "mean": mean_vals.mean(),
            "std": mean_vals.std()
        }

# Print results
print("\nAnnual mean water level statistics (1996–2024):")
for region, stats in overall_stats.items():
    print(f"{region}: mean = {stats['mean']:.2f} cm, std = {stats['std']:.2f} cm")



Annual mean water level statistics (1996–2024):
West: mean = 13.51 cm, std = 6.74 cm
Central: mean = 16.58 cm, std = 5.90 cm
East: mean = 10.37 cm, std = 5.39 cm


In [35]:
import pandas as pd
import numpy as np

# 1️⃣ Aggregate: mean per station per year
agg_df = df.groupby(["year", "source_file"])["mean_value_NAVD88"].mean().reset_index()

# 2️⃣ Pivot: rows = years, columns = stations
pivot_df = agg_df.pivot(index="year", columns="source_file", values="mean_value_NAVD88")

# Function to compute mean and std of correlations (excluding self-correlations)
def corr_stats(corr_matrix):
    triu_indices = np.triu_indices_from(corr_matrix, k=1)  # upper triangle, no diagonal
    values = corr_matrix.values[triu_indices]
    return values.mean(), values.std()

# 3️⃣ Correlation stats for each subregion
region_corr_summary = {}
for region_name, station_list in zip(["West", "Central", "East"], [west, central, east]):
    # Keep only stations that exist in pivot_df
    stations_in_data = [s for s in station_list if s in pivot_df.columns]
    if not stations_in_data:
        continue
    sub_pivot = pivot_df[stations_in_data].dropna(how="all")
    corr_matrix = sub_pivot.corr()
    mean_corr, std_corr = corr_stats(corr_matrix)
    region_corr_summary[region_name] = {"mean_corr": mean_corr, "std_corr": std_corr}

region_corr_df = pd.DataFrame(region_corr_summary).T
region_corr_df.index.name = "Region"
print("Mean ± std of correlation among stations in each subregion:")
print(region_corr_df)

# 4️⃣ Correlation stats across all stations
all_corr_matrix = pivot_df.corr()
all_mean_corr, all_std_corr = corr_stats(all_corr_matrix)
print("\nMean ± std of correlation among all stations:")
print(f"Mean: {all_mean_corr:.3f}, Std: {all_std_corr:.3f}")


Mean ± std of correlation among stations in each subregion:
         mean_corr  std_corr
Region                      
West      0.796578  0.155327
Central   0.854939  0.108755
East      0.690681  0.193139

Mean ± std of correlation among all stations:
Mean: 0.794, Std: 0.154


In [21]:
import pandas as pd
import numpy as np

# 1️⃣ Aggregate: mean per station per year
agg_df = df.groupby(["year", "source_file"])["mean_value_NAVD88"].mean().reset_index()

# 2️⃣ Pivot: rows = years, columns = stations
pivot_df = agg_df.pivot(index="year", columns="source_file", values="mean_value_NAVD88")

# Function to compute mean and standard error of correlations (excluding self-correlations)
def corr_stats_se(corr_matrix):
    triu_indices = np.triu_indices_from(corr_matrix, k=1)  # upper triangle, no diagonal
    values = corr_matrix.values[triu_indices]
    n = len(values)
    mean_val = values.mean()
    se_val = values.std(ddof=1) / np.sqrt(n)  # standard error
    return mean_val, se_val

# 3️⃣ Correlation stats for each subregion
region_corr_summary = {}
for region_name, station_list in zip(["West", "Central", "East"], [west, central, east]):
    # Keep only stations that exist in pivot_df
    stations_in_data = [s for s in station_list if s in pivot_df.columns]
    if not stations_in_data:
        continue
    sub_pivot = pivot_df[stations_in_data].dropna(how="all")
    corr_matrix = sub_pivot.corr()
    mean_corr, se_corr = corr_stats_se(corr_matrix)
    region_corr_summary[region_name] = {"mean_corr": mean_corr, "se_corr": se_corr}

region_corr_df = pd.DataFrame(region_corr_summary).T
region_corr_df.index.name = "Region"
print("Mean ± SE of correlation among stations in each subregion:")
print(region_corr_df)

# 4️⃣ Correlation stats across all stations
all_corr_matrix = pivot_df.corr()
all_mean_corr, all_se_corr = corr_stats_se(all_corr_matrix)
print("\nMean ± SE of correlation among all stations:")
print(f"Mean: {all_mean_corr:.3f}, SE: {all_se_corr:.3f}")


Mean ± SE of correlation among stations in each subregion:
         mean_corr   se_corr
Region                      
West      0.797193  0.021213
Central   0.854323  0.014759
East      0.690681  0.064380

Mean ± SE of correlation among all stations:
Mean: 0.793, SE: 0.008


In [ ]:
def check_stations_exist(df, station_list):
    source_files = df['source_file'].unique()
    missing = [s for s in station_list if s not in source_files]
    if missing:
        print(f"Missing stations: {missing}")
    else:
        print("All stations are present.")

# Check each group
print("West stations:")
check_stations_exist(combined_df, west)

print("\nCentral stations:")
check_stations_exist(combined_df, central)

print("\nEast stations:")
check_stations_exist(combined_df, east)

In [ ]:
# Combine all stations into a single set
all_stations = set(west + central + east)

# Find any source_file not in the combined lists
extra_stations = combined_df.loc[~combined_df['source_file'].isin(all_stations), 'source_file'].unique()

if len(extra_stations) > 0:
    print("Source files not in west, central, or east lists:")
    print(extra_stations)
else:
    print("All source_file values are included in the lists.")

In [ ]:
import pandas as pd

# Combine all station lists
all_stations = west + central + east

# Filter DataFrame by source_file names
df_filtered = combined_df[combined_df['source_file'].isin(all_stations)]




In [ ]:
import pandas as pd

# Define the time range
start_year = 1996
end_year = 2024

# Filter df_filtered for years 1996-2024
df_timeseries = df_filtered[(df_filtered['year'] >= start_year) & (df_filtered['year'] <= end_year)]

# Find which stations have any data in this period
stations_with_timeseries = df_timeseries['source_file'].unique()
print("Stations with time series between 1996-2024:")
print(stations_with_timeseries)

all_stations = west + central + east

# Find stations from your list that do NOT have data in 1996-2024
missing_stations = [s for s in all_stations if s not in stations_with_timeseries]

if missing_stations:
    print("Stations from the list missing data in 1996-2024:")
    print(missing_stations)
else:
    print("All stations in the list have data in 1996-2024.")



Backup

In [ ]:
csv_path = "/Volumes/Malonelab/Research/Whitezone/Hydro_Salinity/Hydro/Merged/WZ_hydro_sites_latlon_merged.csv"

df = pd.read_csv(csv_path)

def ddmmss_to_degree(value):
    if pd.isna(value):
        return np.nan

    value = float(value)

    degrees = int(value // 10000)
    minutes = int((value % 10000) // 100)
    seconds = value % 100

    return degrees + minutes / 60.0 + seconds / 3600.0

# =========================
# Apply conversion
# =========================
df["LAT_deg"] = df["Lat"].apply(ddmmss_to_degree)
df["LON_deg"] = -df["Lon"].apply(ddmmss_to_degree)

output_path = "/Volumes/Malonelab/Research/Whitezone/Hydro_Salinity/Hydro/Merged/WZ_hydro_sites_latlon_deg.csv"
df.to_csv(output_path, index=False)

print("Conversion completed.")

In [ ]:
points = gpd.read_file("/Volumes/MaloneLab/Research/Whitezone/Result/Spatial_Modelling/Results/combined_hydro.shp")
# # Ensure CRS is geographic (lat/lon), if not reproject to WGS84
# if points.crs != "EPSG:4326":
#     points = points.to_crs("EPSG:4326")

# # Extract lat/lon
# points["lon"] = points.geometry.x
# points["lat"] = points.geometry.y

# # Now you have a DataFrame with lat/lon
# print(points[["lat", "lon"]].head())

# # 2. Read transects shapefile
# transects = gpd.read_file("/Volumes/MaloneLab-1/Research/Whitezone/Spatial_Data/transects_39_61.shp")

# # Reproject if needed to match CRS
# if transects.crs != "EPSG:4326":
#     transects = transects.to_crs("EPSG:4326")

# print(transects.head())

In [ ]:
import pandas as pd

# Load salinity data
hydrdo_df = pd.read_csv(
    '/Volumes/Malonelab/Research/Whitezone/Hydro_Salinity/Hydro/Merged/annual_summary_merged.csv'
)

# Sites in the CSV
csv_sites = set(salinity_df['source_file'].unique())

# Define station groups
west = [
    "NP-P37", 
    "Upstream_Taylor",
    "ENPLM",
    "ENPTR",
    "NP-146",
    "NP-CHP",
    "JBTS",
    "ENPTC",
    "NP-OL", 
    "SWEVER5A",
    "MUD_CRKM",
]

central = [
    "EVER7", 
    "CT50R", 
    "EVER6", 
    "CT27R", 
    "EVER8",
    "ENPLS",
    "ENPDK",
    "ENPBS",
    "ENPHC",
    "NP-EV6",
    "NP-EP1"
]

east = [
     "SWEVER1",
     "S197_H",
     "S197_T",
     "S20_H",
     "S20_T"
]

# Make sure site column is clean
csv_sites = set(
    salinity_df['source_file']
    .dropna()          # remove NaN
    .astype(str)       # ensure all strings
)

group_sites = set(west + central + east)

# Comparisons
csv_only = csv_sites - group_sites
group_only = group_sites - csv_sites

print(f"Sites in CSV but not in West/Central/East groups ({len(csv_only)}):")
print(sorted(csv_only))

print("\nSites in West/Central/East groups but not in CSV:")
print(sorted(group_only))



In [ ]:
## check if the list is the same as the lat/lon list ###
csv_path = "/Volumes/Malonelab/Research/Whitezone/Hydro_Salinity/Hydro/Merged/WZ_hydro_sites_latlon_deg.csv"
df = pd.read_csv(csv_path)

# Clean Name column
csv_names = set(
    df['Name']
    .dropna()
    .astype(str)
    .str.strip()
)
group_names = set(west + central + east)

# In CSV but NOT in any group
csv_only = csv_names - group_names

# In groups but NOT in CSV
group_only = group_names - csv_names

print(f"Names in CSV but not in West/Central/East groups ({len(csv_only)}):")
print(sorted(csv_only))

print("\nNames in West/Central/East groups but not in CSV:")
print(sorted(group_only))


In [ ]:
# Check if any Name is duplicated
has_duplicates = df['Name'].duplicated().any()
print("Any duplicated Name?:", has_duplicates)

# Show duplicated Names (if any)
duplicated_names = df.loc[df['Name'].duplicated(keep=False), 'Name'].sort_values()
print("Duplicated Names:")
print(duplicated_names)

In [ ]:
# 

# dbhydro_df = dbhydro_df.merge(
#     datum_df[['Name', 'Datum', 'Conversion']],
#     left_on='source_file',
#     right_on='Name',
#     how='left'
# )

# # Drop the redundant 'Name' column from the merge
# dbhydro_df.drop(columns=['Name'], inplace=True)

# # Preview result
# print(dbhydro_df.head())

# # Convert mean_value to NAVD88 only when Datum is '29'
# dbhydro_df['mean_value_NAVD88'] = dbhydro_df.apply(
#     lambda row: row['mean_value'] + row['Conversion'] if str(row['Datum']).strip() == '29' else row['mean_value'],
#     axis=1
# )

# # Preview the result

# dbhydro_df['mean_value_NAVD88'] = dbhydro_df['mean_value_NAVD88'] * 30.48

# combined_df = pd.concat([dbhydro_df, hydro_df], join='inner', ignore_index=True)